# CYP3A4 post-prediction refinement and selection

(c) Byron DeLaBarre / The Consulting Biochemist 2026

This notebook uses `cofolder_processer.py` It is imported here so ensure that the
module beside this notebook (or anywhere on `sys.path`).

**One config object replaces every scattered path variable.** Everything
derives from values setup in the '.env' file: 
`predictions_root` + `run_name`, so `cif_dir`, `pdb_dir`,
`refine_dir`, `work_dir`, `best_dir` and all five manifests are computed, not
restated. `LIGAND_K` is defined in exactly one place.

**`refine_dir` is named after a hash of the refinement parameters** (`cfg.tag`),
so changing `ligand_k`, `wall_k` or anything else in `REFINE_PARAMS` writes to
a new directory to avoid writing over previous efforts.

Driver is required because orbmol & openmm have different environement requirements:
and `orbmol` needs torch 2.14.1+cu132 while `openmm` uses an older torch setup. 

Runthis notebook in `openmm`; the `score` stage shells out to `orbmol`
automatically. 

One-time setup:

```
conda run -n openmm python -m pip install posebusters
```

**Mixed binder / non-binder runs.** 
Allows for different settings applied to different compounds.
For Cyp3A4 this was Heme-Fe binders vs non-binders.
Set `binders` and `nonbinders` (or point
`class_csv` at a classification table) and the arm is resolved *per compound*
inside the refinement: each structure gets its own wall radii and its own
Fe-restraint decision, while everything else — one `cif_dir`, one
`refine_dir`, one score pass, one `best_poses` — stays single. 

Stage order: `convert → refine → validate → score → select → plot`.
Selection ranks by ΔE (generated in orbmol), breaks ties inside `de_tol` by ipTM (generated by co-folding routine), 
Final decision made by clean run on critical QC checks in PoseBusters.

In [ ]:
# ============================================================================
# SETUP -- modules, arm split, run registry, and every config used downstream.
# Also contains variables for setting up pooled run - similar to the cyp_suite_pools.ipynb
# Historical inclusion - it was developed here and then reproduced in the (untested) cyp_suite_pools.ipynb
# ============================================================================
import sys, os, re, shutil, importlib, dataclasses
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import cofolder_processor as cofo_proc
importlib.reload(cofo_proc)
import pandas as pd
import numpy as np
pd.set_option("display.width", 220)

# ============================================================================
# Compound split example 
# Provides example on how to provide different refinement settings for groups that are 
# chemically distinct.  In this case - the ability to create a bond with the Heme-Fe.
#
# Naming is forgiving: cmpd_12, cmpd12, CMPD-12, 12, OCNT-2313038, or
# ocnt_2313038 all resolve to the same compound, and the two lists may mix
# notations freely. Everything is normalised to cmpd_N against the challenge
# csv before use, so cfg.tag does not change if you rewrite the lists in a
# different notation.
#
# ============================================================================
BINDERS = ("cmpd_0", "cmpd_1", "cmpd_5", "cmpd_6", "cmpd_7", "cmpd_9",
           "cmpd_10", "cmpd_11", "cmpd_12", "cmpd_13", "cmpd_17")
NONBINDERS = ("cmpd_2", "cmpd_3", "cmpd_4", "cmpd_8", "cmpd_14", "cmpd_15",
              "cmpd_16", "cmpd_18", "cmpd_19")
ARM_CSV = ""          # a table instead of the lists: cmpd,arm columns

if not ARM_CSV:
    assert BINDERS and NONBINDERS, "fill in BINDERS and NONBINDERS, or set ARM_CSV"
assert not set(BINDERS) & set(NONBINDERS), "a compound is in both arms"

# ---------------------------------------------------------------------------
# WHICH PREDICTION SET.
# ---------------------------------------------------------------------------
OF3_ROOT = #<path to OF3 predictions>#

# examples of how past runs information can be stored for future use.  
# Often we are making small changes that are otherwise hard to keep track of!
RUNS = {
    "of_run4":  dict(run_name="openfold3_cyp3a4_run4_combined",
                     engine="openfold3", predictions_root=OF3_ROOT),
    "of_run2":  dict(run_name="openadmet20_run2", engine="openfold3",
                     predictions_root=OF3_ROOT),
    "run9":     dict(run_name="esmfold2_cyp3a4_run9",  engine="esmfold2"),
    "run5":     dict(run_name="esmfold2_cyp3a4_run5",  engine="esmfold2"),
    "run11":    dict(run_name="esmfold2_cyp3a4_run11", engine="esmfold2"),
}
ACTIVE = "run11"  # set the current active run

# ---------------------------------------------------------------------------
# SHELL -- written down ONCE. The variant names derive from it, so changing
# the shell cannot collide with an earlier variant's score_params.json stamp.
# ---------------------------------------------------------------------------
SHELL = 3.5  # this is the orbmol shell radius for inclusion of atoms.  Using orbmol beyond 400 atoms is outside its training radius
_SH          = f"{SHELL:g}".replace(".", "p")     # 3.5 -> "3p5"
VARIANT      = f"shell{_SH}"  # allows us to generate different directories - very useful when sweeping through values to optimize settings
POOL_VARIANT = f"pooled_shell{_SH}"

# refinement params: these + the arm map define cfg.tag and therefore refine_dir
REFINE_KW = dict(
    arm="mixed", binders=BINDERS, nonbinders=NONBINDERS,
    strict_arms=True, class_csv=ARM_CSV,
    ligand_k=15.0, ligand_tol=0.30, wall_k=1.0e5,
    # dict, NOT the float 0.26: a float pins both arms and hashes to a
    # DIFFERENT tag (c51e0ec4) than the dict (ebcf32cf), even though the
    # radii it produces are identical. ebcf32cf is where the work is.
    wall_rc_default={"nonbinders": 0.26},
)
# scoring/selection params: NOT in the tag, so they live in `variant` instead
SCORE_KW = dict(shell=SHELL, de_tol=0.20, require_pb_pass=True)


def make_run_for(key, **extra):
    """A config for any RUNS entry, carrying only the refinement params.

    That is deliberate. Those fix cfg.tag and therefore refine_dir. Pooling
    reads refine.csv and validate.csv, which live in base_stage_dir and are
    variant-independent, so the shell and variant of a SOURCE run are
    irrelevant to pooling -- the pooled run does its own scoring.
    """
    kw = dict(REFINE_KW); kw.update(extra)
    return cofo_proc.RunConfig.from_env(**RUNS[key], **kw)


# --- the ordinary single-run config ----------------------------------------
RUN = cofo_proc.RunConfig.from_env(**RUNS[ACTIVE], **REFINE_KW, **SCORE_KW,
                            variant=VARIANT)
cfg = of3 = RUN        # aliases, so older cells referring to either still work

# --- the pooled run --------------------------------------------------------
POOL_ESM_KEY = "run9"
POOL_OF3_KEY = "of_run2"
POOL_NAME    = f"pooled_{POOL_ESM_KEY}_{POOL_OF3_KEY}"
SEED_OFFSET  = 0       # raise only if Cell 2 reports key collisions

RUN_ESM = make_run_for(POOL_ESM_KEY)
RUN_OF3 = make_run_for(POOL_OF3_KEY)

RUN_POOL = cofo_proc.RunConfig.from_env(
    run_name=POOL_NAME,
    engine="esmfold2",            # pooled files use the ESMFold2 convention
    **REFINE_KW, **SCORE_KW,
    variant=POOL_VARIANT,
)
RUN_POOL.iptm_csv = str(RUN_POOL.run_dir / "iptm_ranknorm.csv")
RUN_POOL.ensure_dirs()
RUN_POOL.refine_dir.mkdir(parents=True, exist_ok=True)

# --- what we are about to work with ----------------------------------------
print(f"{len(BINDERS)} binders + {len(NONBINDERS)} non-binders = "
      f"{len(BINDERS) + len(NONBINDERS)}   shell={SHELL}\n")
print(f"ACTIVE  {ACTIVE}")
print(RUN.describe())

print(f"\n--- pooled sources ---")
for lbl, c in (("ESM", RUN_ESM), ("OF3", RUN_OF3)):
    n_full = len(list(c.refine_dir.glob("*_full.pdb")))
    ok = all(c.manifest(s).is_file() for s in ("refine", "validate"))
    print(f"{lbl}  {c.run_name:<30} tag={c.tag}  {n_full:4d} poses  "
          f"manifests {'OK' if ok else 'MISSING'}")
    print(f"     {c.refine_dir}")
print(f"\nPOOL {RUN_POOL.refine_dir}")
print(f"iptm {RUN_POOL.iptm_csv}")

In [ ]:
# Pooling between two co-folding outputs (eg OF3 and ESMFold2).  See the cyp_suite_pools.ipynb for a more detailed explanation of the pooling procedure.
#  ============================================================================
# Assemble the pooled run: copy both generators' refined poses into one
# refine_dir, merge their refine/validate manifests, and write a
# rank-normalised ipTM file.
#
# Poses are enumerated from refine.csv, not score.csv -- the pooled run does
# its own scoring, so the source runs' dE values are never used and their
# shell/variant settings are irrelevant here.
#
# ipTM is rank-normalised WITHIN each generator because OF3 and ESMFold2
# confidence values are not on a common scale. 1.0 = that generator's best
# pose for the compound, 0.0 = its worst.
# ============================================================================
DRY_RUN = False

def _remap(stem, off):
    return re.sub(r"_seed_(\d+)_", lambda m: f"_seed_{int(m.group(1))+off}_", stem)

plan, refine_rows, val_rows, iptm_rows = [], [], [], []

for gen, rcfg, off in (("esm", RUN_ESM, SEED_OFFSET), ("of3", RUN_OF3, 0)):
    RF = pd.read_csv(rcfg.manifest("refine"))
    VA = pd.read_csv(rcfg.manifest("validate"))
    IP = cofo_proc._load_iptm(rcfg)                 # handles both file layouts

    if "skipped" in RF.columns:
        RF = RF[~RF["skipped"].fillna(False).astype(bool)]
    keep = RF[["cmpd", "seed", "sample"]].drop_duplicates()
    print(f"{gen}: {len(keep):4d} refined poses, {keep.cmpd.nunique()} compounds")

    # rank-normalised ipTM over this generator's full pose set
    IP = IP.copy()
    IP["_rank"] = IP.groupby("cmpd")["iptm"].rank(ascending=False, method="min")
    _n = IP.groupby("cmpd")["iptm"].transform("size")
    IP["iptm_q"] = np.where(_n > 1, 1.0 - (IP["_rank"] - 1) / (_n - 1), 1.0)

    k = keep.merge(RF, on=["cmpd", "seed", "sample"], how="left")
    for r in k.itertuples():
        for col in ("full", "submission"):
            src = Path(getattr(r, col))
            if not src.is_file():
                print(f"  [!] missing {col}: {src}"); continue
            plan.append((src, RUN_POOL.refine_dir / _remap(src.name, off)))

    for df, sink in ((k, refine_rows),
                     (keep.merge(VA, on=["cmpd","seed","sample"], how="left"), val_rows),
                     (keep.merge(IP, on=["cmpd","seed","sample"], how="left"), iptm_rows)):
        d = df.copy()
        d["seed"] = d["seed"].astype(int) + off
        d["generator"] = gen
        sink.append(d)

RFM = pd.concat(refine_rows, ignore_index=True)
VAM = pd.concat(val_rows,    ignore_index=True)
IPM = pd.concat(iptm_rows,   ignore_index=True)

dups = RFM.duplicated(["cmpd", "seed", "sample"]).sum()
print(f"\n{len(plan)} files to copy, {len(RFM)} poses, {dups} key collisions")
print(RFM.groupby("generator").size().to_string())

if dups:
    print("  [!] raise SEED_OFFSET; refusing")
elif DRY_RUN:
    print("  DRY_RUN -- nothing written. Set DRY_RUN = False to apply.")
else:
    for s, d in plan:
        shutil.copy2(s, d)
    RFM.to_csv(RUN_POOL.manifest("refine"), index=False)
    VAM.to_csv(RUN_POOL.manifest("validate"), index=False)
    IPM[["cmpd", "seed", "sample", "iptm_q"]].rename(
        columns={"iptm_q": "iptm"}).to_csv(RUN_POOL.iptm_csv, index=False)
    n_pdb = len(list(RUN_POOL.refine_dir.glob("*_full.pdb")))
    print(f"  {n_pdb} *_full.pdb in {RUN_POOL.refine_dir}")
    print(f"  wrote refine.csv, validate.csv, {Path(RUN_POOL.iptm_csv).name}")

In [ ]:
# ============================================================================
# Score the pooled set on one common pocket per compound, then select.
# dE is comparable across generators ONLY because every pose here is scored
# against the same union pocket -- which is why the pooled run rescores
# rather than merging the source score.csv files.
# ============================================================================
res_pool = cofo_proc.run_all(RUN_POOL, stages=("score", "select"))

RFM = pd.read_csv(RUN_POOL.manifest("refine"))
SC  = pd.read_csv(RUN_POOL.manifest("score")).merge(
          RFM[["cmpd","seed","sample","generator"]],
          on=["cmpd","seed","sample"], how="left")
S   = res_pool["select"].merge(RFM[["cmpd","seed","sample","generator"]],
                               on=["cmpd","seed","sample"], how="left")

print(f"\n=== picks by generator (shell={SHELL}) ===")
print(S["generator"].value_counts().to_string())

print("\n=== pocket size / atom-limit check ===")
chk = (SC.groupby("cmpd")[["n_pocket_res","n_atoms_complex"]].max()
         .join(SC.groupby("cmpd")["over_max_atoms"].any()))
print(chk.to_string())
if chk["over_max_atoms"].any():
    print("  [!] some complexes exceeded max_atoms; those energies are "
          "less trustworthy")

print("\n=== dE by generator ===")
print(SC.groupby("generator")["dE"].agg(["count","min","median","mean"]).round(3))

print("\n=== selection ===")
print(S[["cmpd","generator","arm","seed","dE","dE_min","dE_spread","iptm",
         "decided_by","stereo_ok","pb_critical_pass"]].to_string(index=False))
print(f"\nbest poses -> {RUN_POOL.best_dir}")

In [ ]:
# Copy the top-N refined poses per generator per compound into the pooled
# refine_dir, with ESMFold2 seeds offset so (cmpd, seed, sample) stays unique.
# Also merges refine.csv (carries arm + stereo verdicts into selection) and
# validate.csv (avoids re-running PoseBusters), and writes the rank-normalised
# ipTM file.
DRY_RUN = False

def _remap(stem, off):
    return re.sub(r"_seed_(\d+)_", lambda m: f"_seed_{int(m.group(1))+off}_", stem)

plan, iptm_rows, refine_rows, val_rows = [], [], [], []
for tag, rcfg, off in (("esm", RUN_ESM, SEED_OFFSET), ("of3", RUN_OF3, 0)):
    SC = pd.read_csv(rcfg.manifest("score"))
    RF = pd.read_csv(rcfg.manifest("refine"))
    VA = pd.read_csv(rcfg.manifest("validate"))
    IP = cofo_proc._load_iptm(rcfg)                     # handles both file layouts

    # rank-normalised ipTM over the generator's FULL pose set, before pruning
    IP = IP.copy()
    IP["rank"] = IP.groupby("cmpd")["iptm"].rank(ascending=False, method="min")
    n = IP.groupby("cmpd")["iptm"].transform("size")
    IP["iptm_q"] = np.where(n > 1, 1.0 - (IP["rank"] - 1) / (n - 1), 1.0)

    keep = (SC.sort_values("dE").groupby("cmpd").head(TOP_N)
              [["cmpd", "seed", "sample"]])
    print(f"{tag}: {len(SC)} scored -> {len(keep)} kept "
          f"({SC.cmpd.nunique()} compounds)")

    k = keep.merge(RF, on=["cmpd", "seed", "sample"], how="left")
    for r in k.itertuples():
        for col in ("full", "submission"):
            src = Path(getattr(r, col))
            if not src.is_file():
                print(f"  [!] missing {col}: {src}"); continue
            plan.append((src, RUN_POOL.refine_dir / _remap(src.name, off)))
    for df, sink in ((k, refine_rows), (keep.merge(VA, on=["cmpd","seed","sample"],
                                                   how="left"), val_rows),
                     (keep.merge(IP, on=["cmpd","seed","sample"], how="left"),
                      iptm_rows)):
        d = df.copy()
        d["seed"] = d["seed"].astype(int) + off
        d["generator"] = tag
        sink.append(d)

RFM = pd.concat(refine_rows, ignore_index=True)
VAM = pd.concat(val_rows,    ignore_index=True)
IPM = pd.concat(iptm_rows,   ignore_index=True)

dups = RFM.duplicated(["cmpd", "seed", "sample"]).sum()
print(f"\n{len(plan)} files to copy, {len(RFM)} poses, {dups} key collisions")
if dups:
    print("  [!] raise SEED_OFFSET; refusing")
elif DRY_RUN:
    print("  DRY_RUN -- nothing written. Set DRY_RUN = False to apply.")
else:
    for s, d in plan:
        shutil.copy2(s, d)
    RFM.to_csv(RUN_POOL.manifest("refine"), index=False)
    VAM.to_csv(RUN_POOL.manifest("validate"), index=False)
    IPM[["cmpd", "seed", "sample", "iptm_q"]].rename(
        columns={"iptm_q": "iptm"}).to_csv(RUN_POOL.iptm_csv, index=False)
    print(f"  wrote refine/validate manifests and {RUN_POOL.iptm_csv}")
    print(IPM.groupby("generator")["iptm_q"].describe()[["count","min","50%","max"]])

In [ ]:
res_pool = cofo_proc.run_all(RUN_POOL, stages=("score", "select"))

S = res_pool["select"]
P = pd.read_csv(RUN_POOL.manifest("score")).merge(
        RFM[["cmpd","seed","sample","generator"]], on=["cmpd","seed","sample"],
        how="left")
S = S.merge(P[["cmpd","seed","sample","generator"]],
            on=["cmpd","seed","sample"], how="left")
print(S["generator"].value_counts().to_string())
print()
print(S[["cmpd","generator","arm","seed","dE","dE_min","dE_spread","iptm",
         "decided_by","stereo_ok","pb_critical_pass"]].to_string(index=False))

In [ ]:
RF = pd.read_csv(RUN_POOL.manifest("refine"))          # carries `generator`
SC = pd.read_csv(RUN_POOL.manifest("score")).merge(
        RF[["cmpd","seed","sample","generator"]],
        on=["cmpd","seed","sample"], how="left")

g = SC.groupby(["cmpd","generator"])["dE"].agg(["count","min","median"]).unstack()
print(g.round(3).to_string())
print()
print(SC.groupby("generator")["dE"].agg(["count","min","median","mean"]).round(3))

In [ ]:
import dataclasses

RUN_POOL_EQ = dataclasses.replace(RUN_POOL, variant="pooled_eq")
RUN_POOL_EQ.ensure_dirs()

RF = pd.read_csv(RUN_POOL.manifest("refine"))          # carries `generator`
SC = pd.read_csv(RUN_POOL.manifest("score")).merge(
        RF[["cmpd","seed","sample","generator"]],
        on=["cmpd","seed","sample"], how="left")

# equal numbers per generator, per compound (cmpd_5 has 2 ESM poses, cmpd_15 has 4)
n_by  = SC.groupby(["cmpd","generator"])["dE"].size().unstack()
n_per = n_by.min(axis=1).astype(int)
print(n_per.to_string())

SC = SC.sort_values("dE")
SC["_rk"]  = SC.groupby(["cmpd","generator"]).cumcount()
SC["_lim"] = SC["cmpd"].map(n_per)
# median-centre dE within (compound, generator), using the FULL within-generator
# set for the location estimate, then subsample equally.
SC["dE_raw"] = SC["dE"]
SC["dE_med"] = SC.groupby(["cmpd","generator"])["dE_raw"].transform("median")
SC["dE"]     = SC["dE_raw"] - SC["dE_med"]      # what select will rank on

SC = SC.sort_values("dE")
SC["_rk"]  = SC.groupby(["cmpd","generator"]).cumcount()
SC["_lim"] = SC["cmpd"].map(n_per)
K = SC[SC["_rk"] < SC["_lim"]].drop(columns=["_rk","_lim"])

print(K.groupby("generator")["dE"].agg(["count","min","median"]).round(3))
K.drop(columns=["generator","dE_raw","dE_med"]).to_csv(
    RUN_POOL_EQ.manifest("score"), index=False)

K.drop(columns=["generator"]).to_csv(RUN_POOL_EQ.manifest("score"), index=False)
print(f"\n{len(K)} poses kept: {K.groupby('generator').size().to_dict()}")

sel_eq = cofo_proc.run_all(RUN_POOL_EQ, stages=("select",))["select"]
sel_eq = sel_eq.merge(RF[["cmpd","seed","sample","generator"]],
                      on=["cmpd","seed","sample"], how="left")
print()
print(sel_eq["generator"].value_counts().to_string())
print()
print(sel_eq[["cmpd","generator","arm","seed","dE","dE_min","dE_spread",
              "iptm","decided_by","stereo_ok","pb_critical_pass"]].to_string(index=False))

In [ ]:
P = pd.read_csv(RUN_POOL.manifest("score"))
print(P.groupby("cmpd")[["n_pocket_res","n_atoms_complex"]].max()
       .join(P.groupby("cmpd")["over_max_atoms"].any()).to_string())

In [ ]:
# Example on getting output from moving ambiguous compounds between binder and non-binder sets.
# --- the arm-assignment experiment -----------------------------------------
# The three cyanopyridines. Chemically ambiguous: their pyridine N is
# deactivated by CN (and Br, on cmpd_0), so whether they are genuine type-II
# binders is a judgment call. RUN_NEW tests them as non-binders.
TRIO = ("cmpd_0", "cmpd_5", "cmpd_9")
BINDERS_NEW    = tuple(c for c in BINDERS if c not in TRIO)
NONBINDERS_NEW = tuple(NONBINDERS) + TRIO

def make_run(binders, nonbinders):
    """Identical in every respect except the arm map."""
    return cofo_proc.RunConfig.from_env(
        **RUNS[ACTIVE],

        # --- mixed arms ----------------------------------------------------
        arm         = "mixed",
        binders     = binders,
        nonbinders  = nonbinders,
        strict_arms = True,
        class_csv   = ARM_CSV,

        # --- refinement: these + the arm map define cfg.tag ----------------
        ligand_k        = 15.0,
        ligand_tol      = 0.30,
        wall_k          = 1.0e5,
        # dict, NOT the float 0.26: a float pins both arms and hashes to a
        # DIFFERENT tag (c51e0ec4) than the dict (ebcf32cf), even though the
        # radii it produces are identical. ebcf32cf is where the work is.
        wall_rc_default = {"nonbinders": 0.26},

        # --- scoring / selection -------------------------------------------
        shell           = 4.5,        # converged on the shell sweep
        de_tol          = 0.20,
        require_pb_pass = True,
        variant         = "shell4p5",
    )

RUN_OLD = make_run(BINDERS, NONBINDERS)          # trio as binders (current)
RUN_NEW = make_run(BINDERS_NEW, NONBINDERS_NEW)  # trio as non-binders

RUN = cfg = of3 = RUN_OLD      # aliases, so older cells still work

In [ ]:
print(f"\n  RUN_OLD tag {RUN_OLD.tag}  ->  {RUN_OLD.refine_dir.name}")
print(f"  RUN_NEW tag {RUN_NEW.tag}  ->  {RUN_NEW.refine_dir.name}")
assert RUN_OLD.tag != RUN_NEW.tag, "arm change did not alter the tag"

print(f"{len(BINDERS)} binders + {len(NONBINDERS)} non-binders = "
      f"{len(BINDERS) + len(NONBINDERS)}\n")
print(RUN.describe())
ip = RUN.run_dir / "harvest_iptm.csv"
print(f"\nACTIVE  {ACTIVE}")
print(f"  cif   {len(list(RUN.cif_dir.glob('*.cif'))):4d}  {RUN.cif_dir}")
print(f"  pdb   {len(list(RUN.pdb_dir.glob('*.pdb'))):4d}  {RUN.pdb_dir}")
print(f"  iptm  {'found' if ip.is_file() else 'MISSING':5s} {ip}")
print(f"  chal  {'found' if Path(RUN.challenge_csv).is_file() else 'MISSING'}")

In [ ]:
TRIO = ("cmpd_0", "cmpd_5", "cmpd_9")

res_new = cofo_proc.run_all(
    RUN_NEW,
    stages = ("convert", "refine", "validate", "score", "select"),
    refine = dict(only=TRIO),
    score  = dict(only=TRIO),
)

In [ ]:
import pandas as pd
from pathlib import Path

TRIO = ["cmpd_0", "cmpd_5", "cmpd_9"]
ARM_RUNS = {"binder_arm":    RUN_OLD,   # trio refined as binders (current)
            "nonbinder_arm": RUN_NEW}   # trio refined as non-binders
OUT  = Path("arm_test_trio.csv")

keep = {
 "refine": ["cmpd","seed","sample","arm","min_fe_dist","heme_dist_min",
            "through_ring","clash_min","clash_all_min","e_min","lig_rmsd","skipped"],
 "score":  ["cmpd","seed","sample","dE","n_pocket_res","n_atoms_complex",
            "E_complex","E_pocket","E_ligand"],
 "select": ["cmpd","arm","seed","sample","dE","dE_min","dE_spread","iptm",
            "stereo_ok","n_stereo_ok","pb_critical_pass","same_as_dE_only","file"],
}

frames = []
for label, rcfg in ARM_RUNS.items():
    for stage, cols in keep.items():
        p = Path(rcfg.manifest(stage))
        if not p.is_file():
            print(f"  [!] {label}/{stage}: {p} missing"); continue
        d = pd.read_csv(p)
        d = d[d["cmpd"].isin(TRIO)]
        have = [c for c in cols if c in d.columns]
        miss = [c for c in cols if c not in d.columns]
        if miss:
            print(f"  [i] {label}/{stage}: no column {miss}")
        d = d[have].copy()
        d.insert(0, "stage", stage)
        d.insert(0, "run", label)
        frames.append(d)

out = pd.concat(frames, ignore_index=True)
out.to_csv(OUT, index=False)
print(f"\n{len(out)} rows -> {OUT.resolve()}")
print(out.to_string(index=False))

In [ ]:
# Take input from OF3 output -- either two arm directories or one combined
# directory -- and merge into a single folder ready for intake by cyp_suite.
# ============================================================================
# Strips the "cmpd_<N>__" prefix and writes into DEST, concatenating the
# harvest_iptm.csv file(s).
#
# TWO-DIRECTORY MODE (SRC_SINGLE = None): cross-checks the directory split
# against cfg's BINDERS/NONBINDERS. The arm used in refinement comes from your
# lists, NOT from which directory a file sat in -- so if the two disagree,
# that silently refines a compound the wrong way. This refuses rather than
# letting that happen.
#
# SINGLE-DIRECTORY MODE (SRC_SINGLE set): the directory carries no arm
# information, so there is nothing to cross-check. The arm still comes from
# your lists. This refuses only if a compound is in neither list.
# ============================================================================
import re, shutil
from pathlib import Path

# --- set SRC_SINGLE to use one pre-combined directory; leave None for two ---
SRC_SINGLE = Path("/home/byron/projects/openadmet/cyp/openfold_work/"
                  "cyp_predictions/openadmet20_run2/all_predictions_tmp")

SRC_DIRS  = {"binders":    Path("/path/to/binders/all_predictions_tmp"),
             "nonbinders": Path("/path/to/nonbinders/all_predictions_tmp")}
DEST      = of3.cif_dir
MODE      = "copy"                 # "copy" keeps the originals; "move" doesn't
DRY_RUN   = False
IPTM_NAME = "harvest_iptm.csv"

DEST = Path(DEST)
DEST.mkdir(parents=True, exist_ok=True)     # also creates the run dir above it
SOURCES = ({None: Path(SRC_SINGLE)} if SRC_SINGLE is not None
           else {k: Path(v) for k, v in SRC_DIRS.items()})
print(f"mode        : {'single combined directory' if SRC_SINGLE is not None else 'two arm directories'}")
print(f"destination : {DEST}")
print(f"ipTM target : {DEST.parent / IPTM_NAME}\n")

PREFIX = re.compile(r"^(cmpd_\d+)__(?=cmpd_\d+_seed_)")
amap = of3.arm_map() if of3.is_mixed else {}
plan, problems, seen, frames = [], [], {}, []

for arm, src in SOURCES.items():
    label = arm or "combined"
    if not src.is_dir():
        problems.append(f"{label}: {src} is not a directory")
        if src.parent.is_dir():       # help find the right name
            near = sorted(d.name for d in src.parent.iterdir() if d.is_dir())
            problems.append(f"        directories in {src.parent}: {near[:8]}")
        continue
    found = set()
    for p in sorted(src.glob("*.cif")):
        name = PREFIX.sub("", p.name)
        m = cofo_proc.parse_pose(name, of3)
        if not m:
            problems.append(f"{p.name}: does not parse"); continue
        found.add(m["cmpd"])
        key = (m["cmpd"], m["seed"], m["sample"])
        if key in seen:
            problems.append(
                f"{key} appears twice in {src}" if arm is None else
                f"{key} appears in both {seen[key]} and {arm}")
            continue
        seen[key] = label
        plan.append((p, DEST / name, amap.get(m["cmpd"], label), m["cmpd"]))
    if amap:
        if arm is None:
            # no directory arm to check against; just confirm every compound
            # is covered by one of your lists
            orphan = sorted(c for c in found if c not in amap)
            if orphan:
                problems.append(f"not in BINDERS or NONBINDERS: {orphan}")
        else:
            wrong = sorted(c for c in found if amap.get(c) != arm)
            if wrong:
                problems.append(f"{arm} dir holds compounds your lists call "
                                f"{[amap.get(c) for c in wrong]}: {wrong}")
    ip = src / IPTM_NAME
    if ip.is_file():
        d = pd.read_csv(ip); d["_arm"] = label; frames.append(d)
    else:
        problems.append(f"{label}: no {IPTM_NAME} in {src}")

by_arm = {}
for _o, _n, arm, c in plan:
    by_arm.setdefault(arm, set()).add(c)
print(f"{len(plan)} files -> {DEST}")
for arm, cc in sorted(by_arm.items()):
    print(f"    {arm:11s} {len(cc):2d} compounds, "
          f"{sum(1 for x in plan if x[2] == arm)} poses")
for q in problems:
    print(f"  [!] {q}")

if problems:
    print("  refusing; nothing written")
elif DRY_RUN:
    print("  DRY_RUN -- nothing written. Set DRY_RUN = False to apply.")
else:
    for o, n, _a, _c in plan:
        (shutil.copy2 if MODE == "copy" else shutil.move)(str(o), str(n))
    IP = pd.concat(frames, ignore_index=True)
    _key = ["Model"] if "Model" in IP.columns else ["cmpd", "seed", "sample"]
    IP = IP.drop_duplicates(_key).drop(columns=["_arm"])
    IP.to_csv(DEST.parent / IPTM_NAME, index=False)
    print(f"  merged {len(IP)} ipTM rows -> {DEST.parent / IPTM_NAME}")

In [ ]:
# PRE-FLIGHT CHECK
# ============================================================================
# PRE-FLIGHT -- check the arm assignment against what is actually on disk
# BEFORE starting a refinement that takes an hour. Costs two seconds.
# ============================================================================
seen = {}
for p in sorted(cfg.pdb_dir.glob("*.pdb")):
    m = cofo_proc.parse_pose(p, cfg)
    if m:
        seen.setdefault(m["cmpd"], []).append(p.name)

amap = cfg.arm_map()                 # canonical cmpd_N keys, however written

print("--- resolved arm assignment ---")
print(cfg.arm_table().to_string(index=False))
print()
missing_from_disk = sorted(set(amap) - set(seen))
unassigned       = sorted(set(seen) - set(amap))

print(f"--- {len(seen)} compounds / {sum(len(v) for v in seen.values())} "
      f"poses in {cfg.pdb_dir} ---")
for c in sorted(seen, key=lambda s: int(s.split("_")[1])):
    print(f"  {c:9s} {len(seen[c])} poses   arm="
          f"{amap.get(c, '** UNASSIGNED **')}")

if unassigned:
    print(f"\n[!] in pdb_dir but in neither arm list: {unassigned}")
    print("    stage_refine will raise. Add them to BINDERS / NONBINDERS.")
if missing_from_disk:
    print(f"\n[!] assigned an arm but no poses on disk: {missing_from_disk}")
    print("    these will be absent from best_poses -- fold them first.")
if not unassigned and not missing_from_disk:
    print("\nOK: every compound on disk has an arm, and every assigned "
          "compound has poses.")


In [ ]:
# generic run all command
cofo_proc.run_all(RUN, stages=("convert", "refine", "validate", "score", "select", "plot"))

In [ ]:
import pandas as pd
sc = pd.read_csv(RUN.manifest("score"))
print(sc[["n_pocket_res","n_atoms_complex","n_caps","charge_pocket","charge_lig"]].nunique())
print()
print(sc.groupby("cmpd")[["n_pocket_res","n_atoms_complex"]].agg(["min","max","nunique"]))

In [ ]:
# Optimized setting run.  Not much different from previous,
#  but conceptually a larger sphere in orbmol (still within training set range) and looser constraints for HEME wall for non-binders is better

import dataclasses

run9 = dataclasses.replace(cfg, shell=4.5, variant="shell4p5",
                           wall_rc_default={"nonbinders": 0.26})

print(run9.describe())
cofo_proc.sweep_plan([run9])
cofo_proc.run_all(run9, stages=("convert", "refine", "validate", "score", "select", "plot"))
print(run9.best_dir)

In [ ]:
cofo_proc.compare_selections(cfgs)         # did the CHOICE change?
cofo_proc.compare_best_structures(cfgs)    # did the COORDINATES change?   <- the one that matters

In [ ]:
# check runs created during sweep to get tags numbers (attached to output directory)
for c in cfgs:
    print(f"rc {c.wall_rc_default_for('nonbinders'):.2f}  tag {c.tag}  {c.refine_dir.name}")

In [ ]:
# Quick summary of changes due to parameter sweep
#importlib.reload(cs)
base = dataclasses.replace(cfg, shell=4.5, variant="shell4p5")
cfgs = list(cofo_proc.sweep(base, grid))     # tags are deterministic, so these
                                      # resolve to the same directories
print(cofo_proc.compare_runs(cfgs)[["variant","rc_nonbind","through_ring","heme_min",
                             "heme_lt30","pb_critical_pass","rmsd_max",
                             "dE_med"]].to_string(index=False))

In [ ]:
# ============================================================================
# dE PROFILES ACROSS A SWEEP -- does a wider shell re-rank the poses?
#
# One panel per compound. Poses are ordered along x by the BASELINE variant's
# dE, so the baseline line is monotonic by construction and any other variant
# that disagrees shows up as a line that stops rising. Y is dE minus each
# variant's own minimum for that compound: absolute energies are not
# comparable between shells (different atom counts), the ranking is.
#
# Filled marker = the pose that variant actually selected.
# ============================================================================
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
%matplotlib inline
CFGS      = cfgs              # from cofo_proc.sweep(...), baseline first
NCOL      = 5
RELATIVE  = True              # False to plot raw dE instead
OUT       = None              # Path to save a PNG, or None

labels = [c.variant or c.tag for c in CFGS]
vals   = [c.shell for c in CFGS]

S = {}
for c, lab in zip(CFGS, labels):
    f = c.manifest("score")
    if not f.is_file():
        print(f"[!] no score manifest for {lab}"); continue
    d = pd.read_csv(f).drop_duplicates(["cmpd", "seed", "sample"])
    sel = c.manifest("select")
    picks = {}
    if sel.is_file():
        picks = {r["cmpd"]: (int(r["seed"]), int(r["sample"]))
                 for r in pd.read_csv(sel).to_dict("records")}
    S[lab] = (d, picks)
if not S:
    raise SystemExit("no score manifests found")

labels = [l for l in labels if l in S]
base   = labels[0]
cmpds  = sorted(S[base][0].cmpd.unique(),
                key=lambda s: int(str(s).split("_")[-1]))

# sequential single-hue ramp: shell is ordinal, so magnitude -> light to dark
ramp = plt.get_cmap("Blues")(np.linspace(0.42, 0.95, len(labels)))
GRID, INK, MUTED = "#E3E6EA", "#1F2328", "#6B7280"

nrow = int(np.ceil(len(cmpds) / NCOL))
fig, axes = plt.subplots(nrow, NCOL, figsize=(2.75 * NCOL, 2.5 * nrow),
                         sharex=False, squeeze=False)
changed = []
for i, cm in enumerate(cmpds):
    ax = axes[i // NCOL][i % NCOL]
    g0 = S[base][0].query("cmpd == @cm").sort_values("dE")
    order = [(int(r["seed"]), int(r["sample"])) for r in g0.to_dict("records")]
    if not order:
        ax.axis("off"); continue
    x = np.arange(len(order))
    picks_here = set()
    for lab, col in zip(labels, ramp):
        d, picks = S[lab]
        m = {(int(r["seed"]), int(r["sample"])): r["dE"]
             for r in d.query("cmpd == @cm").to_dict("records")}
        y = np.array([m.get(k, np.nan) for k in order], float)
        if np.all(np.isnan(y)):
            continue
        if RELATIVE:
            y = y - np.nanmin(y)
        ax.plot(x, y, "-", lw=1.6, color=col, zorder=3,
                marker="o", ms=3.0, mew=0)
        p = picks.get(cm)
        if p in order:
            j = order.index(p)
            picks_here.add(p)
            ax.plot([j], [y[j]], "o", ms=8, color=col, zorder=5,
                    mec="white", mew=1.6)
    if len(picks_here) > 1:
        changed.append(cm)
    ax.set_title(cm + ("   *" if len(picks_here) > 1 else ""),
                 fontsize=9, color=INK,
                 fontweight="bold" if len(picks_here) > 1 else "normal")
    ax.grid(True, axis="y", color=GRID, lw=0.7, zorder=0)
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color(GRID)
    ax.tick_params(labelsize=7.5, colors=MUTED, length=3)
    ax.set_xticks(x[::max(1, len(x)//5)])
    ax.set_xticklabels([f"{order[k][0]}" for k in range(0, len(order),
                                                        max(1, len(x)//5))])
for j in range(len(cmpds), nrow * NCOL):
    axes[j // NCOL][j % NCOL].axis("off")

fig.supxlabel("pose (contact set), ordered by the baseline's dE",
              fontsize=9, color=MUTED, y=0.048)
fig.supylabel("dE - min(dE) per variant  [eV]" if RELATIVE else "dE  [eV]",
              fontsize=9, color=MUTED)
fig.suptitle(f"dE profiles across the shell sweep   "
             f"({len(changed)}/{len(cmpds)} compounds change pick, marked *)",
             fontsize=11, color=INK, y=0.995)
fig.legend(handles=[Line2D([], [], color=c, lw=2.4,
                           label=f"shell {v} A" if v is not None else l)
                    for l, v, c in zip(labels, vals, ramp)],
           loc="lower center", ncol=min(len(labels), 6), frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, 0.002))
fig.tight_layout(rect=[0.02, 0.085, 1, 0.97])
if OUT:
    fig.savefig(OUT, dpi=160, bbox_inches="tight", facecolor="white")
    print("wrote", OUT)
plt.show()

print(f"\n{len(changed)}/{len(cmpds)} compounds change pick across the sweep")
if changed:
    print("  " + ", ".join(changed))
else:
    print("  the shell does not affect selection at all -- 3.5 A is sufficient")

In [ ]:
# ============================================================================
# dE PROFILES ACROSS A SWEEP -- does a wider shell re-rank the poses?
#
# One panel per compound. Poses are ordered along x by the BASELINE variant's
# dE, so the baseline line is monotonic by construction and any other variant
# that disagrees shows up as a line that stops rising. Y is dE minus each
# variant's own minimum for that compound: absolute energies are not
# comparable between shells (different atom counts), the ranking is.
#
# Filled marker = the pose that variant actually selected.
# ============================================================================
import json
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# cofolder_processor's plot stage used to call matplotlib.use("Agg"), which is global
# and permanent for the kernel -- after it, plt.show() draws nothing and says
# nothing. Restore an inline backend if that happened.
if mpl.get_backend().lower() in ("agg", "template"):
    try:
        mpl.use("module://matplotlib_inline.backend_inline")
        print(f"[plot] backend was Agg (nothing would render); "
              f"switched to {mpl.get_backend()}")
    except Exception:
        print("[plot] backend is Agg and inline is unavailable -- the figure "
              "will be SAVED but not displayed; see OUT below")

CFGS      = cfgs              # from cofo_proc.sweep(...), baseline first
NCOL      = 5
RELATIVE  = True              # False to plot raw dE instead
OUT       = "sweep_dE_profiles.png"   # always written, so you
                              # get the figure even if the
                              # backend cannot display it

# label by whatever parameter actually varies across the sweep
def _sweep_labels(cfgs):
    from dataclasses import fields, asdict
    base = asdict(cfgs[0])
    varied = [f.name for f in fields(cfgs[0])
              if len({json.dumps(asdict(c)[f.name], sort_keys=True,
                                 default=str) for c in cfgs}) > 1
              and f.name not in ("variant", "refine_subdir")]
    if not varied:
        return [c.variant or c.tag for c in cfgs], None
    key = varied[0]
    out = []
    for c in cfgs:
        v = asdict(c)[key]
        if isinstance(v, dict):
            v = "/".join(f"{k[:2]} {x}" for k, x in sorted(v.items()))
        elif v is None:
            v = "default"
        out.append(f"{key} {v}")
    return out, key

labels, SWEPT = _sweep_labels(CFGS)
print(f"[plot] sweeping {SWEPT or '(nothing varies -- labelling by tag)'}")
vals   = labels

S = {}
for c, lab in zip(CFGS, labels):
    f = c.manifest("score")
    if not f.is_file():
        print(f"[!] no score manifest for {lab}"); continue
    d = pd.read_csv(f).drop_duplicates(["cmpd", "seed", "sample"])
    sel = c.manifest("select")
    picks = {}
    if sel.is_file():
        picks = {r["cmpd"]: (int(r["seed"]), int(r["sample"]))
                 for r in pd.read_csv(sel).to_dict("records")}
    S[lab] = (d, picks)
if not S:
    raise SystemExit("no score manifests found")

labels = [l for l in labels if l in S]
base   = labels[0]
cmpds  = sorted(S[base][0].cmpd.unique(),
                key=lambda s: int(str(s).split("_")[-1]))

# sequential single-hue ramp: shell is ordinal, so magnitude -> light to dark
ramp = plt.get_cmap("Blues")(np.linspace(0.42, 0.95, len(labels)))
GRID, INK, MUTED = "#E3E6EA", "#1F2328", "#6B7280"

nrow = int(np.ceil(len(cmpds) / NCOL))
fig, axes = plt.subplots(nrow, NCOL, figsize=(2.75 * NCOL, 2.5 * nrow),
                         sharex=False, squeeze=False)
changed = []
for i, cm in enumerate(cmpds):
    ax = axes[i // NCOL][i % NCOL]
    g0 = S[base][0].query("cmpd == @cm").sort_values("dE")
    order = [(int(r["seed"]), int(r["sample"])) for r in g0.to_dict("records")]
    if not order:
        ax.axis("off"); continue
    x = np.arange(len(order))
    picks_here = set()
    for lab, col in zip(labels, ramp):
        d, picks = S[lab]
        m = {(int(r["seed"]), int(r["sample"])): r["dE"]
             for r in d.query("cmpd == @cm").to_dict("records")}
        y = np.array([m.get(k, np.nan) for k in order], float)
        if np.all(np.isnan(y)):
            continue
        if RELATIVE:
            y = y - np.nanmin(y)
        ax.plot(x, y, "-", lw=1.6, color=col, zorder=3,
                marker="o", ms=3.0, mew=0)
        p = picks.get(cm)
        if p in order:
            j = order.index(p)
            picks_here.add(p)
            ax.plot([j], [y[j]], "o", ms=8, color=col, zorder=5,
                    mec="white", mew=1.6)
    if len(picks_here) > 1:
        changed.append(cm)
    ax.set_title(cm + ("   *" if len(picks_here) > 1 else ""),
                 fontsize=9, color=INK,
                 fontweight="bold" if len(picks_here) > 1 else "normal")
    ax.grid(True, axis="y", color=GRID, lw=0.7, zorder=0)
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color(GRID)
    ax.tick_params(labelsize=7.5, colors=MUTED, length=3)
    ax.set_xticks(x[::max(1, len(x)//5)])
    ax.set_xticklabels([f"{order[k][0]}" for k in range(0, len(order),
                                                        max(1, len(x)//5))])
for j in range(len(cmpds), nrow * NCOL):
    axes[j // NCOL][j % NCOL].axis("off")

fig.supxlabel("pose (contact set), ordered by the baseline's dE",
              fontsize=9, color=MUTED, y=0.048)
fig.supylabel("dE - min(dE) per variant  [eV]" if RELATIVE else "dE  [eV]",
              fontsize=9, color=MUTED)
fig.suptitle(f"dE profiles across the {SWEPT or ''} sweep   "
             f"({len(changed)}/{len(cmpds)} compounds change pick, marked *)",
             fontsize=11, color=INK, y=0.995)
fig.legend(handles=[Line2D([], [], color=c, lw=2.4, label=l)
                    for l, c in zip(labels, ramp)],
           loc="lower center", ncol=min(len(labels), 6), frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, 0.002))
fig.tight_layout(rect=[0.02, 0.085, 1, 0.97])
if OUT:
    fig.savefig(OUT, dpi=160, bbox_inches="tight", facecolor="white")
    print(f"wrote {OUT}  (backend: {mpl.get_backend()})")
plt.show()

print(f"\n{len(changed)}/{len(cmpds)} compounds change pick across the sweep")
if changed:
    print("  " + ", ".join(changed))
else:
    print(f"  {SWEPT} does not change any pick.")
    print("  NOTE: for a REFINEMENT parameter the structures were still")
    print("  re-minimised, so identical picks are DIFFERENT coordinates.")
    print("  Use cofo_proc.compare_best_structures(cfgs) to see how far they moved.")

In [ ]:
print(cofo_proc.compare_runs(cfgs)[["variant","shell","atoms_med","atoms_max",
                             "over_max","dE_med","n_picked"]].to_string(index=False))

In [ ]:
S = pd.read_csv(cfgs[-1].manifest("score"))      # the 6.0 A variant
print(S.groupby("cmpd").n_atoms_complex.max().sort_values(ascending=False).head(5))

In [ ]:
cofo_proc.run_stage(run9, "select")

In [ ]:
# ============================================================================
# Inspect the manifests. All five are keyed on (cmpd, seed, sample).
#
# POINT THIS AT THE CONFIG YOU MEAN. `cfg` is the BASE config from cell 2 --
# if you have been working with a derived one (a variant, a different shell,
# a swept wall radius), `cfg` still resolves to the base directories and you
# will be reading an older run without noticing.
# ============================================================================
INSPECT = run11          # e.g. INSPECT = run9   or   INSPECT = cfgs[2]

print("=" * 78)
print(f"INSPECTING  {INSPECT.run_name}   tag {INSPECT.tag}"
      f"   variant {INSPECT.variant or '-'}")
print(f"  shell {INSPECT.shell}   rc(nonbind) "
      f"{INSPECT.wall_rc_default_for('nonbinders')}   rc(bind) "
      f"{INSPECT.wall_rc_default_for('binders')}   de_tol {INSPECT.de_tol}")
print(f"  refine {INSPECT.refine_dir.name}/{INSPECT.stage_dir.name}")
print(f"  best   {INSPECT.best_dir.name}")
print("=" * 78)

for name in ("convert", "refine", "validate", "score", "select"):
    p = INSPECT.manifest(name)
    print(f"{name:9s} {'exists' if p.is_file() else 'missing':8s} {p}")

REF = pd.read_csv(INSPECT.manifest("refine"))
SC  = pd.read_csv(INSPECT.manifest("score"))
PB  = pd.read_csv(INSPECT.manifest("validate"))
SEL = pd.read_csv(INSPECT.manifest("select"))

print("\n--- refinement geometry ---")
print(REF[["structure", "arm", "lig_rmsd", "clash_all_min", "heme_dist_min",
           "min_fe_dist", "through_ring"]].round(2).to_string(index=False))

print("\n--- geometry by arm: heme contact should separate cleanly ---")
print(REF.groupby("arm")[["lig_rmsd", "heme_dist_min", "min_fe_dist",
                          "clash_all_min"]]
        .agg(["median", "min", "max"]).round(2).to_string())

print("\n--- selection ---")
print(SEL[["cmpd", "arm", "seed", "decided_by", "n_tried", "dE", "iptm",
           "pb_critical_pass", "file"]].round(3).to_string(index=False))
if "decided_by" in SEL:
    n_ip = int((SEL.decided_by == "iptm").sum())
    print(f"  decided by dE {len(SEL)-n_ip}, by ipTM {n_ip}"
          + ("   [!] ipTM is doing most of the deciding -- try "
             "cofo_proc.preview_de_tol(INSPECT)" if n_ip > len(SEL) / 2 else ""))

print("\n--- completeness of the single best_poses directory ---")
cofo_proc.audit_best(INSPECT)

print("\n--- where PoseBusters disagreed with dE ---")
m = SC.merge(PB[["cmpd", "seed", "sample", "pb_critical_pass"]],
             on=["cmpd", "seed", "sample"], how="left")
m["rank"] = m.groupby("cmpd")["dE"].rank(method="first").astype(int)
bad1 = m[(m["rank"] == 1) & (m.pb_critical_pass == False)]
if len(bad1):
    print("rank-1 poses that fail critical checks (selection moved on):")
    print(bad1[["cmpd", "seed", "dE"]].round(3).to_string(index=False))
else:
    print("every rank-1 pose passes")

In [ ]:
# ============================================================================
# dE AND ipTM PROFILES, one run -- inline, rescalable.
#
# Per compound: two stacked panels sharing the x-axis (poses ranked by dE,
# best first).  TOP  dE - best, with the de_tol line.
#                BOTTOM ipTM for the same poses, in the same order.
#
# Read them together. dE rises left-to-right by construction, so if ipTM's
# peak is NOT at x=1, ipTM wanted a different pose than the energy did --
# and inside the de_tol band it gets its way.
#
# No twin y-axis: two measures on one axis would let the apparent agreement
# be set by the scaling rather than the data.
# ============================================================================
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

CFG      = RUN            # which run to plot
YMAX     = None           # eV on the dE panels; None = auto
IPTM_LIM = None           # e.g. (0.85, 1.0) to zoom the ipTM panels
NCOL     = 5
PANEL    = (2.7, 2.6)     # inches per compound (both panels together)
SHOW     = None           # None = all; else ["cmpd_0", "cmpd_7"]
SAVE     = None

if mpl.get_backend().lower() in ("agg", "template"):
    try:
        mpl.use("module://matplotlib_inline.backend_inline")
    except Exception:
        print("[plot] backend is Agg; set SAVE to get the figure as a file")

SC = pd.read_csv(CFG.manifest("score")).drop_duplicates(
    ["cmpd", "seed", "sample"])
pbf = CFG.manifest("validate")
if pbf.is_file():
    PB = pd.read_csv(pbf)
    if "pb_critical_pass" in PB.columns:
        SC = SC.merge(PB[["cmpd", "seed", "sample", "pb_critical_pass"]]
                      .drop_duplicates(["cmpd", "seed", "sample"]),
                      on=["cmpd", "seed", "sample"], how="left")
if "pb_critical_pass" not in SC:
    SC["pb_critical_pass"] = np.nan

ip = cofo_proc._load_iptm(CFG)
SC = (SC.merge(ip, on=["cmpd", "seed", "sample"], how="left")
      if ip is not None else SC.assign(iptm=np.nan))
if SC.iptm.isna().all():
    print("[plot] no ipTM available -- the lower panels will be empty")

_sel = CFG.manifest("select")
PICK = ({(r["cmpd"], int(r["seed"]), int(r["sample"])) for r in
         pd.read_csv(_sel).to_dict("records")} if _sel.is_file() else set())

cmpds = sorted(SC.cmpd.unique(), key=lambda s: int(str(s).split("_")[-1]))
if SHOW:
    cmpds = [c for c in cmpds if c in set(SHOW)]
if not cmpds:
    raise SystemExit("no compounds to plot")

INK, MUTED, GRID = "#1F2328", "#6B7280", "#E3E6EA"
DEC, IPC, FAIL, PICKC = "#2E6FB7", "#5B8C5A", "#B6BCC4", "#D1603D"

nrow = int(np.ceil(len(cmpds) / NCOL))
fig = plt.figure(figsize=(PANEL[0] * NCOL, PANEL[1] * nrow))
# nested: the OUTER grid spaces compounds apart (room for titles), the inner
# 2-row grid keeps each compound's dE and ipTM panels tight together.
outer = GridSpec(nrow, NCOL, figure=fig, hspace=0.42, wspace=0.34,
                 left=0.055, right=0.995, top=0.945, bottom=0.075)

n_tied = n_disagree = 0
for i, cm in enumerate(cmpds):
    r, c = divmod(i, NCOL)
    inner = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[r, c],
                                    height_ratios=[2.0, 1.1], hspace=0.10)
    ax  = fig.add_subplot(inner[0])
    axi = fig.add_subplot(inner[1], sharex=ax)
    g = SC[SC.cmpd == cm].sort_values("dE")
    if g.empty:
        ax.axis("off"); axi.axis("off"); continue
    y = g.dE.values; best = y.min(); yy = y - best
    x = np.arange(1, len(yy) + 1)
    ok = g.pb_critical_pass.fillna(True).astype(bool).values
    it = g.iptm.values

    ax.plot(x, yy, "-", lw=1.5, color=DEC, zorder=2)
    ax.plot(x[ok], yy[ok], "o", ms=3.2, color=DEC, mew=0, zorder=3)
    if (~ok).any():
        ax.plot(x[~ok], yy[~ok], "o", ms=3.2, color=FAIL, mew=0, zorder=3)
    ax.axhline(CFG.de_tol, ls="--", lw=1.1, color=PICKC, alpha=.75, zorder=1)
    tied = int((yy <= CFG.de_tol).sum()); n_tied += tied > 1

    if np.isfinite(it).any():
        axi.plot(x, it, "-", lw=1.4, color=IPC, zorder=2)
        axi.plot(x, it, "o", ms=3.0, color=IPC, mew=0, zorder=3)
        j_ip = int(np.nanargmax(it))
        if j_ip > 0 and yy[j_ip] <= CFG.de_tol:
            n_disagree += 1
        axi.plot([x[j_ip]], [it[j_ip]], "o", ms=6.5, color=IPC,
                 mec="white", mew=1.3, zorder=4)

    for j, rec in enumerate(g.to_dict("records")):
        if (rec["cmpd"], int(rec["seed"]), int(rec["sample"])) in PICK:
            ax.plot([x[j]], [yy[j]], "o", ms=8.0, color=PICKC,
                    mec="white", mew=1.4, zorder=5)
            if np.isfinite(it[j]):
                axi.plot([x[j]], [it[j]], "o", ms=8.0, color=PICKC,
                         mec="white", mew=1.4, zorder=5)
            break

    ax.set_title(f"{cm}   best {best:+.2f} eV   tied {tied}/{len(yy)}",
                 fontsize=8.5, color=INK)
    for a in (ax, axi):
        a.grid(True, axis="y", color=GRID, lw=.7); a.set_axisbelow(True)
        for sp in ("top", "right"):
            a.spines[sp].set_visible(False)
        for sp in ("left", "bottom"):
            a.spines[sp].set_color(GRID)
        a.tick_params(labelsize=7, colors=MUTED, length=3)
    if YMAX is not None:
        ax.set_ylim(-0.02, YMAX)
    if IPTM_LIM:
        axi.set_ylim(*IPTM_LIM)
    plt.setp(ax.get_xticklabels(), visible=False)
    if c == 0:
        ax.set_ylabel("dE - best (eV)", fontsize=7.5, color=MUTED)
        axi.set_ylabel("ipTM", fontsize=7.5, color=MUTED)

fig.suptitle(f"{CFG.run_name}   {CFG.arm_label}   tag={CFG.tag}   "
             f"shell={CFG.shell} A   "
             f"({n_tied}/{len(cmpds)} tied inside de_tol={CFG.de_tol}; "
             f"ipTM peaks away from rank 1 in {n_disagree})",
             fontsize=10.5, color=INK, y=0.997)
fig.supxlabel("pose, ranked by dE (best first)", fontsize=9,
              color=MUTED, y=0.034)
fig.legend(handles=[
    Line2D([], [], color=DEC, marker="o", lw=1.5, ms=4, label="dE"),
    Line2D([], [], color=IPC, marker="o", lw=1.4, ms=4, label="ipTM"),
    Line2D([], [], color=FAIL, marker="o", lw=0, ms=5,
           label="fails critical PoseBusters"),
    Line2D([], [], color=PICKC, marker="o", lw=0, ms=7, label="selected"),
    Line2D([], [], color=PICKC, ls="--", lw=1.1, label="de_tol")],
    loc="lower center", ncol=5, frameon=False, fontsize=8.5,
    bbox_to_anchor=(0.5, 0.002))
if SAVE:
    fig.savefig(SAVE, dpi=160, bbox_inches="tight", facecolor="white")
    print("wrote", SAVE)
plt.show()

print(f"{n_tied}/{len(cmpds)} compounds have >1 pose within de_tol="
      f"{CFG.de_tol} eV of the best.")
print(f"{n_disagree}/{len(cmpds)} have their highest ipTM at a pose that is "
      f"NOT the lowest dE but IS inside de_tol -- these are the compounds "
      f"where ipTM actually overrode the energy.")

In [ ]:
# ============================================================================
# MERGE -- for arms already refined as SEPARATE runs (the two-pass route).
# Not needed if you ran the mixed cfg above, which writes one best_poses
# directly. Use this to combine what you already have on disk.
#
# merge_best raises if the same compound was selected by both runs, because
# that means the arms overlapped -- silently keeping one would hide it.
# ============================================================================
import dataclasses

cfg_b = dataclasses.replace(
    cfg, arm="binders", binders=(), nonbinders=(), class_csv="",
    refine_subdir="PUT_THE_BINDERS_REFINE_DIR_HERE")
cfg_n = dataclasses.replace(
    cfg, arm="nonbinders", binders=(), nonbinders=(), class_csv="",
    refine_subdir="PUT_THE_NONBINDERS_REFINE_DIR_HERE")

for c in (cfg_b, cfg_n):
    print(f"{c.arm_label:11s} {c.best_dir}  "
          f"{len(list(c.best_dir.glob('*.pdb'))) if c.best_dir.is_dir() else 0}"
          f" pdb")

SUBMIT = cfg.run_dir / "submission_merged"
M = cofo_proc.merge_best(SUBMIT, cfg_b, cfg_n, expect=20, strict=True)
print()
print(M.to_string(index=False))


In [ ]:
# ============================================================================
# SWEEP -- one directory per parameter combination, compared at the end.
# Each cfg.tag differs, so no run can overwrite or silently reuse another.
# ============================================================================
grid = {"ligand_k":   [5.0, 15.0],
        "ligand_tol": [0.0, 0.30]}

cfgs = list(cofo_proc.sweep(cfg, grid))
print(f"{len(cfgs)} configs, tags {[c.tag for c in cfgs]}")

for c in cfgs:
    print("\n" + "=" * 70)
    print(c.describe())
    cofo_proc.run_all(c, stages=("refine", "validate", "score"))

print("\n" + "=" * 70)
print(cofo_proc.compare_runs(cfgs).round(3).to_string(index=False))

In [ ]:
# parameter sweep example 2
import dataclasses

base = dataclasses.replace(cfg, shell=4.5, variant="shell4p5_wallrc2p6")

grid = {"wall_rc_default": [None,                    # 0.34, your current setting
                            {"nonbinders": 0.32},
                            {"nonbinders": 0.29},
                            {"nonbinders": 0.26}]}
cfgs = list(cofo_proc.sweep(base, grid))

cofo_proc.sweep_plan(cfgs)            # gate: raises before anything runs

for c in cfgs:
    print(c.describe())
    cofo_proc.run_all(c, stages=("refine", "validate", "score", "select"))

print(cofo_proc.compare_runs(cfgs).to_string(index=False))
T, S = cofo_proc.compare_selections(cfgs)

In [ ]:
#setup to sweep across the orbmol radius
#out = cofo_proc.run_all(cfg, stages=("convert", "refine", "validate",
#                              "score", "select", "plot"))
#cofo_proc.run_all(cfg)

cfgs = list(cofo_proc.sweep(cfg, {"shell": [3.5, 4.5, 5.0, 6.0]}))

for c in cfgs:
    print(c.describe())
    cofo_proc.run_stage(c, "score")
    cofo_proc.run_stage(c, "select")

T, S = cofo_proc.compare_selections(cfgs)
print(T.to_string())
print(cofo_proc.compare_runs(cfgs)[["variant","shell","n_picked","dE_min","dE_med"]]
        .to_string(index=False))


In [ ]:
# Parameter sweep example
import dataclasses

base = dataclasses.replace(cfg, shell=4.5, variant="shell4p5_wallrc2p6")

grid = {"wall_rc_default": [None,                    # 0.34, your current setting
                            {"nonbinders": 0.32},
                            {"nonbinders": 0.29},
                            {"nonbinders": 0.26}]}
cfgs = list(cofo_proc.sweep(base, grid))

cofo_proc.sweep_plan(cfgs)            # gate: raises before anything runs

for c in cfgs:
    print(c.describe())
    cofo_proc.run_all(c, stages=("refine", "validate", "score", "select"))

print(cofo_proc.compare_runs(cfgs).to_string(index=False))
T, S = cofo_proc.compare_selections(cfgs)

In [ ]:
# ============================================================================
# THE ACTIVE RUN. Every stage cell below uses RUN, so switching between
# prediction sets is this one line -- not an edit in four places.
# ============================================================================
RUN = of3                      # or: RUN = cfg   to go back to run 9

print(f"ACTIVE: {RUN.run_name}  engine={RUN.engine}  tag={RUN.tag}"
      f"  variant={RUN.variant or '-'}")
print(f"  cif  {RUN.cif_dir}   ({len(list(RUN.cif_dir.glob('*.cif')))} cifs)")
print(f"  pdb  {RUN.pdb_dir}   ({len(list(RUN.pdb_dir.glob('*.pdb')))} pdbs)")
print(f"  iptm {RUN.run_dir / 'harvest_iptm.csv'}  "
      f"{'found' if (RUN.run_dir / 'harvest_iptm.csv').is_file() else 'MISSING'}")

In [ ]:
# Combined setup for the CYP suite driver
# ============================================================================
# SETUP -- modules, arm split, and the active run config.
# ============================================================================
import sys, os, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import cofolder_processor as cofo_proc
importlib.reload(cofo_proc)
import pandas as pd
pd.set_option("display.width", 220)

# ---------------------------------------------------------------------------
# THE ARM SPLIT -- the one place it is written down.
#
# Naming is forgiving: cmpd_12, cmpd12, CMPD-12, 12, OCNT-2313038 all resolve
# to the same compound, and the lists may mix notations. Everything is
# normalised to cmpd_N against the challenge csv, so cfg.tag does not change
# if you rewrite a list in a different notation.
#
# Existence is strict. A compound in neither list, in BOTH lists, or not in
# the challenge csv raises HERE and names itself -- rather than surfacing an
# hour later as some *other* compound being unassigned. That strictness is
# the point: a compound quietly falling through is how OCNT-2313038 vanished
# from a 20-file submission.
# ---------------------------------------------------------------------------
BINDERS = ("cmpd_0", "cmpd_1", "cmpd_5", "cmpd_6", "cmpd_7", "cmpd_9",
           "cmpd_10", "cmpd_11", "cmpd_12", "cmpd_13", "cmpd_17")
NONBINDERS = ("cmpd_2", "cmpd_3", "cmpd_4", "cmpd_8", "cmpd_14", "cmpd_15",
              "cmpd_16", "cmpd_18", "cmpd_19")
ARM_CSV = ""          # a table instead of the lists: cmpd,arm columns

if not ARM_CSV:
    assert BINDERS and NONBINDERS, "fill in BINDERS and NONBINDERS, or set ARM_CSV"
assert not set(BINDERS) & set(NONBINDERS), "a compound is in both arms"

# ---------------------------------------------------------------------------
# WHICH PREDICTION SET. One line to switch; everything else is shared.
# ---------------------------------------------------------------------------
OF3_ROOT = <OF3_ROOT>

RUNS = {
    "of_run4":  dict(run_name="openfold3_cyp3a4_run4_combined",
                     engine="openfold3", predictions_root=OF3_ROOT),
    "of_run2":  dict(run_name="openadmet20_run2", engine="openfold3",
                     predictions_root=OF3_ROOT),
    "run9":     dict(run_name="esmfold2_cyp3a4_run9",  engine="esmfold2"),
    "run5":     dict(run_name="esmfold2_cyp3a4_run5",  engine="esmfold2"),
    "run11":    dict(run_name="esmfold2_cyp3a4_run11", engine="esmfold2"),
}
ACTIVE = "run11"

RUN = cofo_proc.RunConfig.from_env(
    **RUNS[ACTIVE],

    # --- mixed arms --------------------------------------------------------
    arm         = "mixed",
    binders     = BINDERS,
    nonbinders  = NONBINDERS,
    strict_arms = True,
    class_csv   = ARM_CSV,

    # --- refinement: these + the arm map define cfg.tag --------------------
    ligand_k        = 15.0,
    ligand_tol      = 0.30,
    wall_k          = 1.0e5,
    # dict, NOT the float 0.26: a float pins both arms and hashes to a
    # DIFFERENT tag (c51e0ec4) than the dict (ebcf32cf), even though the
    # radii it produces are identical. ebcf32cf is where the work is.
    wall_rc_default = {"nonbinders": 0.26},

    # --- scoring / selection -----------------------------------------------
    shell           = 3.5,        # converged on the shell sweep
    de_tol          = 0.20,
    require_pb_pass = True,
    variant         = "shell3p5",
)
cfg = of3 = RUN        # aliases, so older cells referring to either still work

print(f"{len(BINDERS)} binders + {len(NONBINDERS)} non-binders = "
      f"{len(BINDERS) + len(NONBINDERS)}\n")
print(RUN.describe())
ip = RUN.run_dir / "harvest_iptm.csv"
print(f"\nACTIVE  {ACTIVE}")
print(f"  cif   {len(list(RUN.cif_dir.glob('*.cif'))):4d}  {RUN.cif_dir}")
print(f"  pdb   {len(list(RUN.pdb_dir.glob('*.pdb'))):4d}  {RUN.pdb_dir}")
print(f"  iptm  {'found' if ip.is_file() else 'MISSING':5s} {ip}")
print(f"  chal  {'found' if Path(RUN.challenge_csv).is_file() else 'MISSING'}")